In [1]:
import pandas as pd

players = pd.read_csv("players.csv")
players.head()

,player_id,name,current_club_id,current_club_name,country_of_citizenship,country_of_birth,city_of_birth,date_of_birth,position,sub_position,...,highest_market_value_in_eur,agent_name,contract_expiration_date,current_club_domestic_competition_id,first_name,last_name,player_code,image_url,last_season,url
0,134354,Ian Raeymaekers,498,Ksc Lokeren,Belgium,Belgium,Aalst,1995-01-30,Attack,Centre-Forward,...,50000.0,NaN,NaN,BE1,Ian,Raeymaekers,ian-raeymaekers,https://img.a.transfermarkt.technology/portrai...,2012,https://www.transfermarkt.co.uk/ian-raeymaeker...
1,99946,Mohamed Camara,1095,Es Troyes Ac,Guinea,Guinea,Conakry,1990-09-20,Attack,Centre-Forward,...,300000.0,NaN,NaN,FR1,Mohamed,Camara,mohamed-camara,https://img.a.transfermarkt.technology/portrai...,2012,https://www.transfermarkt.co.uk/mohamed-camara...
2,76948,Pablo Olivera,979,Moreirense Fc,Uruguay,Uruguay,Melo,1987-12-08,Attack,Centre-Forward,...,600000.0,NaN,NaN,PO1,Pablo,Olivera,pablo-olivera,https://img.a.transfermarkt.technology/portrai...,2012,https://www.transfermarkt.co.uk/pablo-olivera/...
3,108372,Aliosman Aydin,38,Fortuna Dusseldorf,Turkey,Germany,Dormagen,1992-02-06,Attack,Centre-Forward,...,125000.0,NaN,NaN,L1,Aliosman,Aydin,aliosman-aydin,https://img.a.transfermarkt.technology/portrai...,2012,https://www.transfermarkt.co.uk/aliosman-aydin...
4,78820,Jaime Alfonso Ruiz,354,Kv Mechelen,Colombia,Colombia,Cali,1984-01-03,Attack,Centre-Forward,...,1700000.0,NaN,NaN,BE1,Jaime Alfonso,Ruiz,jaime-alfonso-ruiz,https://img.a.transfermarkt.technology/portrai...,2012,https://www.transfermarkt.co.uk/jaime-alfonso-...


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [2]:
players.columns

Index(['player_id', 'name', 'current_club_id', 'current_club_name',
       'country_of_citizenship', 'country_of_birth', 'city_of_birth',
       'date_of_birth', 'position', 'sub_position', 'foot', 'height_in_cm',
       'market_value_in_eur', 'highest_market_value_in_eur', 'agent_name',
       'contract_expiration_date', 'current_club_domestic_competition_id',
       'first_name', 'last_name', 'player_code', 'image_url', 'last_season',
       'url'],
      dtype='object')

In [3]:
players.shape

(28226, 23)

In [4]:
players["current_club_domestic_competition_id"].unique()

array(['BE1', 'FR1', 'PO1', 'L1', 'SC1', 'GR1', 'GB1', 'UKR1', 'DK1',
       'ES1', 'TR1', 'IT1', 'RU1', 'NL1'], dtype=object)

In [5]:
pl_players = players[players["current_club_domestic_competition_id"] == "GB1"]
pl_players.shape

(1985, 23)

In [6]:
pl_players["market_value_in_eur"].isna().sum()

np.int64(552)

In [7]:
pl_players_clean = pl_players.dropna(subset=["market_value_in_eur"])
pl_players_clean.shape

(1433, 23)

In [8]:
pl_players_clean["date_of_birth"].head()

,date_of_birth
10,1984-09-01
11,1988-11-04
17,1987-05-15
18,1989-11-11
19,1986-12-02


In [10]:
pl_players_clean = pl_players_clean.copy()
pl_players_clean["date_of_birth"] = pd.to_datetime(pl_players_clean["date_of_birth"])

In [11]:
today = pd.Timestamp("today")
pl_players_clean["age"] = (today - pl_players_clean["date_of_birth"]).dt.days // 365
pl_players_clean[["name", "date_of_birth", "age"]].head()

,name,date_of_birth,age
10,Kei Kamara,1984-09-01,42
11,Chris Martin,1988-11-04,37
17,Garath McCleary,1987-05-15,39
18,Nick Blackman,1989-11-11,36
19,Adam Le Fondre,1986-12-02,39


In [12]:
appearances = pd.read_csv("appearances.csv")
appearances.head()

,appearance_id,game_id,player_id,player_club_id,player_current_club_id,date,player_name,competition_id,yellow_cards,red_cards,goals,assists,minutes_played
0,2483937_52453,2483937,52453,28095,28095,2014-08-08,Haris Handzic,RU1,0,0,0,0,90
1,2479929_67064,2479929,67064,28095,4128,2014-08-03,Felicio Brown Forbes,RU1,0,0,0,0,90
2,2483937_67064,2483937,67064,28095,4128,2014-08-08,Felicio Brown Forbes,RU1,0,0,0,0,90
3,2484582_67064,2484582,67064,28095,4128,2014-08-13,Felicio Brown Forbes,RU1,0,0,0,0,55
4,2485965_67064,2485965,67064,28095,4128,2014-08-16,Felicio Brown Forbes,RU1,0,0,0,0,90


In [13]:
player_stats = appearances.groupby("player_id")[["goals", "assists", "minutes_played"]].sum().reset_index()
player_stats.head()

,player_id,goals,assists,minutes_played
0,10,24,16,4003
1,26,0,0,5252
2,65,13,2,3424
3,80,0,0,450
4,132,3,2,941


In [14]:
final_data = pl_players_clean.merge(player_stats, on="player_id", how="left")
final_data.shape

(1433, 27)

In [15]:
final_data[["goals", "assists", "minutes_played"]].isna().sum()

,0
goals,350
assists,350
minutes_played,350


In [16]:
final_data = final_data.dropna(subset=["goals", "assists", "minutes_played"])
final_data.shape

(1083, 27)

In [17]:
X = final_data[["age", "goals", "assists", "minutes_played"]]
y = final_data["market_value_in_eur"]

X.head()

,age,goals,assists,minutes_played
81,32,0.0,0.0,238.0
82,33,0.0,0.0,58.0
86,30,0.0,0.0,45.0
87,38,0.0,0.0,445.0
89,30,0.0,0.0,31.0


In [18]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(X_train.shape)
print(X_test.shape)

(866, 4)
(217, 4)


In [19]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)

print("Model trained!")

Model trained!


In [20]:
predictions = model.predict(X_test)

# Compare a few predictions to actual values
comparison = pd.DataFrame({
    "Actual Value": y_test,
    "Predicted Value": predictions
})
comparison.head(10)

,Actual Value,Predicted Value
161,350000.0,-9.008103e+06
698,300000.0,-6.773809e+06
935,600000.0,1.370558e+06
904,25000000.0,2.739923e+07
154,400000.0,6.804447e+06
512,6000000.0,9.153748e+06
1108,55000000.0,2.971184e+07
568,500000.0,1.238637e+06
176,700000.0,2.165680e+06
1170,25000000.0,2.751768e+07


In [21]:
from sklearn.metrics import mean_absolute_error, r2_score

mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print(f"Average error: €{mae:,.0f}")
print(f"R² score: {r2:.2f}")

Average error: €8,595,648
R² score: 0.42


In [22]:
def predict_value(age, goals, assists, minutes_played):
    input_data = pd.DataFrame({
        "age": [age],
        "goals": [goals],
        "assists": [assists],
        "minutes_played": [minutes_played]
    })
    prediction = model.predict(input_data)[0]
    return prediction

# Try it on a made-up player
predicted = predict_value(age=25, goals=10, assists=5, minutes_played=2500)
print(f"Predicted market value: €{predicted:,.0f}")

Predicted market value: €15,617,581


In [23]:
def predict_value_interactive():
    print("Enter the player's stats:")
    age = float(input("Age: "))
    goals = float(input("Goals (career total): "))
    assists = float(input("Assists (career total): "))
    minutes_played = float(input("Minutes played (career total): "))

    input_data = pd.DataFrame({
        "age": [age],
        "goals": [goals],
        "assists": [assists],
        "minutes_played": [minutes_played]
    })
    prediction = model.predict(input_data)[0]
    print(f"\nPredicted market value: €{prediction:,.0f}")

predict_value_interactive()

Enter the player's stats:
Age: 22
Goals (career total): 132
Assists (career total): 12
Minutes played (career total): 433

Predicted market value: €40,390,347


In [24]:
final_data["position"].unique()

array(['Midfield', 'Attack', 'Defender', 'Goalkeeper'], dtype=object)

In [25]:
final_data_encoded = pd.get_dummies(final_data, columns=["position"], drop_first=True)
final_data_encoded.columns

Index(['player_id', 'name', 'current_club_id', 'current_club_name',
       'country_of_citizenship', 'country_of_birth', 'city_of_birth',
       'date_of_birth', 'sub_position', 'foot', 'height_in_cm',
       'market_value_in_eur', 'highest_market_value_in_eur', 'agent_name',
       'contract_expiration_date', 'current_club_domestic_competition_id',
       'first_name', 'last_name', 'player_code', 'image_url', 'last_season',
       'url', 'age', 'goals', 'assists', 'minutes_played', 'position_Defender',
       'position_Goalkeeper', 'position_Midfield'],
      dtype='object')

In [26]:
X = final_data_encoded[["age", "goals", "assists", "minutes_played",
                          "position_Defender", "position_Goalkeeper", "position_Midfield"]]
y = final_data_encoded["market_value_in_eur"]

X.head()

,age,goals,assists,minutes_played,position_Defender,position_Goalkeeper,position_Midfield
81,32,0.0,0.0,238.0,False,False,True
82,33,0.0,0.0,58.0,False,False,True
86,30,0.0,0.0,45.0,False,False,False
87,38,0.0,0.0,445.0,False,False,False
89,30,0.0,0.0,31.0,True,False,False


In [27]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print(f"Average error: €{mae:,.0f}")
print(f"R² score: {r2:.2f}")

Average error: €8,547,795
R² score: 0.42


In [28]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(random_state=42)
rf_model.fit(X_train, y_train)

rf_predictions = rf_model.predict(X_test)

rf_mae = mean_absolute_error(y_test, rf_predictions)
rf_r2 = r2_score(y_test, rf_predictions)

print(f"Random Forest - Average error: €{rf_mae:,.0f}")
print(f"Random Forest - R² score: {rf_r2:.2f}")

Random Forest - Average error: €5,824,132
Random Forest - R² score: 0.59


In [29]:
def predict_value_interactive():
    print("Enter the player's stats:")
    age = float(input("Age: "))
    goals = float(input("Goals (career total): "))
    assists = float(input("Assists (career total): "))
    minutes_played = float(input("Minutes played (career total): "))
    position = input("Position (Attack/Midfield/Defender/Goalkeeper): ")

    input_data = pd.DataFrame({
        "age": [age],
        "goals": [goals],
        "assists": [assists],
        "minutes_played": [minutes_played],
        "position_Defender": [1 if position == "Defender" else 0],
        "position_Goalkeeper": [1 if position == "Goalkeeper" else 0],
        "position_Midfield": [1 if position == "Midfield" else 0]
    })

    prediction = rf_model.predict(input_data)[0]
    print(f"\nPredicted market value: €{prediction:,.0f}")

predict_value_interactive()

Enter the player's stats:
Age: 21
Goals (career total): 123
Assists (career total): 43
Minutes played (career total): 566
Position (Attack/Midfield/Defender/Goalkeeper): Attack

Predicted market value: €106,500,000
